# Kenya Hockey Union — Data Collection

This notebook scrapes match results and event timelines (goals, cards,
penalty corners/strokes) for the 2026 season from the
[Kenya Hockey Union](https://www.kenyahockeyunion.org/) website —
covering **all 8 competitions running in 2026**:

- Premier League Women (PLW) · Premier League Men (PLM)
- Super League Men (SLM) · Super League Women (SLW)
- National League Men — East Zone (NLM-EZ), Central Zone (NLM-CZ),
  West Zone (NLM-WZ), South Zone (NLM-SZ)

Every URL in `SEASON_URLS` was confirmed directly against the live site
(not guessed) by locating a real team page in each competition and
reading its "Standings" link — KHU changes this URL's slug format most
seasons, so guessing would be unreliable.

**Pipeline:** open each competition's season page → discover its match
URLs → scrape every match page for teams, score, and event timeline,
tagged with its real competition code → assemble `matches_df` and
`events_df` across all 8 competitions → save to `data/raw/`.

**Output files:**
- `data/raw/matches_2026.csv` — one row per match, across all competitions
- `data/raw/events_2026.csv` — one row per in-match event (goal/card, minute, player, team)

**Note on reliability:** long scrapes (100+ pages) can cause the underlying Chrome/Selenium session to hang or die partway through — this previously caused every match after the failure point to be silently skipped. The scraper now retries transient failures, automatically restarts the browser if the session dies, and proactively recycles it every 40 matches to avoid this happening at all. Any matches that still fail after 3 retries are collected in `failed_matches` and can be retried on their own via Cell 33B, without re-scraping everything.

**Note on a fixed timing bug (found via real-data verification):** an earlier version of this scraper used a flat 2-second pause before reading each match page, rather than waiting specifically for the event timeline to render. Since the timeline loads asynchronously, this caused it to silently return an empty events list for matches where the timeline simply hadn't finished loading yet — indistinguishable from a genuine scoreless match, but wrong. In one real run, this affected 110 of 130 matches (84.6%). Fixed by waiting explicitly for the timeline element (falling back gracefully after 10s if it truly never appears), plus an inline warning if a nonzero-score match still comes back with zero events after that wait. **If you scraped data before this fix, re-run this notebook fully — every goal-based metric downstream (Top Scorers, Elo, Coach/Scout Intelligence, everything) needs the corrected event data.**

**Update, confirmed via direct verification:** the timing theory above turned out to be incomplete. Most matches on the source site simply have no event timeline at all — confirmed by fetching a real 5-0 match with zero embedded event data — so no amount of waiting recovers them. The wait was shortened from 10s to 3s accordingly: long enough for genuinely slow-loading timelines, but no longer paying a 10-second tax per match on the majority that have nothing to find. This meaningfully speeds up the full scrape without losing any real data.

**Note on data completeness:** matches that finished 0–0 with no cards
have no event timeline on the source site — this is expected, not a
scraping error, and is handled explicitly in `scrape_match()` below.

**Note on URL stability:** if a competition's season rolls over (e.g.
2027) or KHU changes its slug format again, the discovery loop below
will print a clear warning naming the competition rather than silently
scraping nothing — check `SEASON_URLS` first if you see that warning.

**Speed — two things changed to address this directly:**

1. **Headless mode is now on by default** — Chrome no longer renders a visible window, which is a straightforward speed win with no tradeoffs for a scrape this size. Set `HEADLESS = False` in the Configure Chrome cell only if you need to visually debug.

2. **Scraping is now incremental.** On every run, this notebook loads whatever's already saved in `data/raw/matches_2026.csv` and skips any match already there — only genuinely new matches get scraped. This is the real fix for "what happens when there are twice as many matches": your first run scrapes the full season once; every run after that only scrapes the *delta* of newly-played matches, so scrape time stays small and roughly constant regardless of how large the season grows. Set `FULL_RESCRAPE = True` in the Incremental Scraping Setup cell to force a complete re-scrape from zero — worth doing regularly (e.g. weekly) since incremental mode won't detect KHU backfilling event data onto a match that previously had none; it only catches genuinely new matches.

**Requirements:** Chrome + ChromeDriver (managed automatically via
`webdriver-manager`), `selenium`, `beautifulsoup4`, `pandas`.


## Setup

Imports and Chrome WebDriver configuration.

In [1]:
# Cell 1 - Imports

import time
import re
import io
import pandas as pd

from bs4 import BeautifulSoup

from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options

from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

from webdriver_manager.chrome import ChromeDriverManager


In [2]:
import numpy as np

In [3]:
# Cell 2 - Configure Chrome

# Headless mode skips rendering the browser window on screen, which is
# meaningfully faster for a scrape this size (no pixels to paint, no
# window to draw). Set to False only if you need to visually watch the
# browser for debugging — day-to-day runs should keep this True.
HEADLESS = True

chrome_options = Options()

if HEADLESS:
    chrome_options.add_argument("--headless=new")
    chrome_options.add_argument("--window-size=1920,1080")

chrome_options.add_argument("--start-maximized")
chrome_options.add_argument("--disable-blink-features=AutomationControlled")
chrome_options.add_experimental_option("excludeSwitches", ["enable-automation"])
chrome_options.add_experimental_option("useAutomationExtension", False)


def create_driver():
    """
    Builds a fresh Chrome driver. Long scraping runs (100+ pages) can
    cause the underlying ChromeDriver session to time out or die
    partway through (seen as "invalid session id" or a read timeout on
    the local ChromeDriver port) — when that happens every subsequent
    match fails instantly since there's no live browser to talk to.
    This function lets the batch-scraping loop below recreate the
    driver on the fly and keep going instead of losing the rest of
    the run.
    """
    d = webdriver.Chrome(
        service=Service(ChromeDriverManager().install()),
        options=chrome_options
    )

    d.execute_script("""
    Object.defineProperty(navigator, 'webdriver', {
        get: () => undefined
    })
    """)

    return d


driver = create_driver()

print(f"Chrome launched successfully. (headless={HEADLESS})")


Chrome launched successfully. (headless=True)


In [4]:
# Utility Functions

def print_header(title):
    print("=" * 70)
    print(title)
    print("=" * 70)


## Season & Match Discovery

Open each competition's season page and collect the URLs of every individual match.

In [5]:
# Cell 3 - Season URLs

# All 8 competitions running in the 2026 season, confirmed directly from
# kenyahockeyunion.org (each URL was verified by locating a live team page
# for that competition and reading its "Standings" link, since KHU changes
# this URL's slug format almost every season and a guessed one would be
# unreliable — e.g. NLM-EZ used "_-ez-2024-2025" last year and "-nlm-ez-2026"
# this year).
SEASON_URLS = {

    "PLW":
    "https://www.kenyahockeyunion.org/joomsport_season/premier-league-women-plw-2026/",

    "PLM":
    "https://www.kenyahockeyunion.org/joomsport_season/premier-league-men-plm-2026/",

    "SLM":
    "https://www.kenyahockeyunion.org/joomsport_season/super-league-men-slm-2026/",

    "SLW":
    "https://www.kenyahockeyunion.org/joomsport_season/super-league-women-slw-2026/",

    "NLM-EZ":
    "https://www.kenyahockeyunion.org/joomsport_season/national-league-men-_-ez-nlm-ez-2026/",

    "NLM-CZ":
    "https://www.kenyahockeyunion.org/joomsport_season/national-league-men-_-cz-nlm-cz-2026/",

    "NLM-WZ":
    "https://www.kenyahockeyunion.org/joomsport_season/national-league-men-wz-nlm-wz-2026/",

    "NLM-SZ":
    "https://www.kenyahockeyunion.org/joomsport_season/national-league-men-sz-nlm-sz-2026/",

}

SEASON_URLS


{'PLW': 'https://www.kenyahockeyunion.org/joomsport_season/premier-league-women-plw-2026/',
 'PLM': 'https://www.kenyahockeyunion.org/joomsport_season/premier-league-men-plm-2026/',
 'SLM': 'https://www.kenyahockeyunion.org/joomsport_season/super-league-men-slm-2026/',
 'SLW': 'https://www.kenyahockeyunion.org/joomsport_season/super-league-women-slw-2026/',
 'NLM-EZ': 'https://www.kenyahockeyunion.org/joomsport_season/national-league-men-_-ez-nlm-ez-2026/',
 'NLM-CZ': 'https://www.kenyahockeyunion.org/joomsport_season/national-league-men-_-cz-nlm-cz-2026/',
 'NLM-WZ': 'https://www.kenyahockeyunion.org/joomsport_season/national-league-men-wz-nlm-wz-2026/',
 'NLM-SZ': 'https://www.kenyahockeyunion.org/joomsport_season/national-league-men-sz-nlm-sz-2026/'}

In [6]:
# Cell 4-7 - Discover Match URLs Across All Competitions

# Loops over every competition in SEASON_URLS (not just one), opens its
# season page, and collects every match URL on it. Matches are tagged
# with their real competition code here, which is what lets Cell 33
# correctly label every match instead of defaulting everything to one
# competition.
#
# CONFIRMED REAL BUG, FIXED: this previously visited only the default
# standings view of each season URL. That page\'s only match links live
# in each team\'s "Current form" column, which KHU\'s site only ever
# shows the team\'s most recent 5 results in \u2014 confirmed directly by
# checking the real page. Once a team has played more than 5 matches,
# its earliest matches have no link anywhere on that page at all, so
# they were structurally impossible for this scraper to discover, no
# matter how many times it was re-run. This is not a transient miss;
# it gets worse as the season progresses.
#
# Fixed by visiting the separate "?action=calendar" view instead, which
# lists every match all season, and paginating through it properly \u2014
# confirmed directly that KHU paginates this view (25 matches per page
# by default) and that simply requesting a larger page size via the URL
# does not reliably work (the site\'s own pagination is JavaScript-
# driven, not a plain URL parameter), so this pages through using
# Selenium, one real page load at a time, stopping only once a page
# returns no match links it hasn\'t already seen.
#
# Retry + driver recreation kept from the original version, since a
# single network blip mid-pagination should not be allowed to silently
# truncate a competition\'s results.

all_match_links = {}   # competition code -> list of match URLs

MAX_RETRIES_DISCOVERY = 3
MAX_CALENDAR_PAGES = 20   # safety cap; a real season should never need this many

for comp, season_url in SEASON_URLS.items():

    calendar_url = season_url.rstrip("/") + "/?action=calendar"
    print(f"Opening {comp}: {calendar_url}")

    links = set()
    success = False

    for attempt in range(1, MAX_RETRIES_DISCOVERY + 1):
        try:
            page_num = 1
            while page_num <= MAX_CALENDAR_PAGES:
                page_url = f"{calendar_url}&pagejs={page_num}"
                driver.get(page_url)
                time.sleep(6)

                html = driver.page_source
                soup = BeautifulSoup(html, "html.parser")

                links_from_tags = {
                    a["href"] for a in soup.find_all("a", href=True)
                    if "joomsport_match" in a["href"]
                }
                links_from_regex = set(re.findall(
                    r'https://www\.kenyahockeyunion\.org/joomsport_match/[^"]+',
                    html
                ))
                page_links = links_from_tags | links_from_regex

                new_links = page_links - links
                if not new_links:
                    # This page repeated links already seen (or had none) —
                    # confirms we\'ve reached the end of this competition\'s
                    # match list, not a failure.
                    break

                links |= new_links
                page_num += 1

            success = True
            break

        except Exception as e:
            err = str(e)
            session_dead = (
                "ERR_NAME_NOT_RESOLVED" in err
                or "invalid session id" in err
                or "session deleted" in err
                or "chrome not reachable" in err
                or "Read timed out" in err
                or "Connection refused" in err
            )
            if session_dead:
                print(f"  \u26a0 Attempt {attempt}/{MAX_RETRIES_DISCOVERY}: network/browser issue "
                      f"({err.splitlines()[0][:80]}). Restarting driver...")
                try:
                    driver.quit()
                except Exception:
                    pass
                driver = create_driver()
                time.sleep(2)
            else:
                print(f"  \u26a0 Attempt {attempt}/{MAX_RETRIES_DISCOVERY} failed: {err.splitlines()[0][:80]}")
                time.sleep(2)

    if not success:
        print(f"  \u2717 Giving up on {comp} after {MAX_RETRIES_DISCOVERY} attempts \u2014 "
              f"check your internet connection, or re-run this cell later.")
    elif not links:
        print(f"  \u26a0 No match links found for {comp}. "
              f"Check that SEASON_URLS['{comp}'] is the correct, current URL.")
    else:
        print(f"  \u2713 Found {len(links)} matches")

    all_match_links[comp] = sorted(links)

print()
total = sum(len(v) for v in all_match_links.values())
print(f"Total matches discovered across {len(SEASON_URLS)} competitions: {total}")


Opening PLW: https://www.kenyahockeyunion.org/joomsport_season/premier-league-women-plw-2026/?action=calendar


  ✓ Found 30 matches
Opening PLM: https://www.kenyahockeyunion.org/joomsport_season/premier-league-men-plm-2026/?action=calendar


  ✓ Found 52 matches
Opening SLM: https://www.kenyahockeyunion.org/joomsport_season/super-league-men-slm-2026/?action=calendar


  ✓ Found 33 matches
Opening SLW: https://www.kenyahockeyunion.org/joomsport_season/super-league-women-slw-2026/?action=calendar


  ✓ Found 20 matches
Opening NLM-EZ: https://www.kenyahockeyunion.org/joomsport_season/national-league-men-_-ez-nlm-ez-2026/?action=calendar


  ✓ Found 31 matches
Opening NLM-CZ: https://www.kenyahockeyunion.org/joomsport_season/national-league-men-_-cz-nlm-cz-2026/?action=calendar


  ✓ Found 3 matches
Opening NLM-WZ: https://www.kenyahockeyunion.org/joomsport_season/national-league-men-wz-nlm-wz-2026/?action=calendar


  ✓ Found 11 matches
Opening NLM-SZ: https://www.kenyahockeyunion.org/joomsport_season/national-league-men-sz-nlm-sz-2026/?action=calendar


  ✓ Found 6 matches

Total matches discovered across 8 competitions: 186


In [7]:
# DIAGNOSTIC - Why Zero Match Links Found

# Run this on ONE competition (PLM) to see exactly what Selenium
# actually captured, rather than guessing at a fix. This checks three
# things separately: whether the raw page source even contains the
# text "joomsport_match" at all (confirming whether Selenium captured
# the content), how long the captured page is (a suspiciously short
# page suggests the page didn't finish loading), and what BeautifulSoup
# itself finds when asked directly for every <a> tag on the page.

test_url = SEASON_URLS["PLM"]
print("Testing:", test_url)

driver.get(test_url)
time.sleep(8)

html = driver.page_source
print(f"\nCaptured page length: {len(html)} characters")
contains_match_text = "joomsport_match" in html
print(f"Does raw page source contain 'joomsport_match' anywhere?: {contains_match_text}")

soup = BeautifulSoup(html, "html.parser")
all_links = soup.find_all("a", href=True)
print(f"\nTotal <a> tags with an href found by BeautifulSoup: {len(all_links)}")

match_links_found = [a["href"] for a in all_links if "joomsport_match" in a["href"]]
print(f"Of those, how many contain 'joomsport_match': {len(match_links_found)}")

if match_links_found:
    print("\nFirst 3 match links found:")
    for link in match_links_found[:3]:
        print(" ", link)
else:
    print("\nNo match links found by BeautifulSoup. Showing the first 5 real hrefs")
    print("captured on the page, to see what IS actually there:")
    for a in all_links[:5]:
        print(" ", a["href"])


Testing: https://www.kenyahockeyunion.org/joomsport_season/premier-league-men-plm-2026/



Captured page length: 195853 characters
Does raw page source contain 'joomsport_match' anywhere?: True

Total <a> tags with an href found by BeautifulSoup: 160
Of those, how many contain 'joomsport_match': 50

First 3 match links found:
  https://www.kenyahockeyunion.org/joomsport_match/premier-league-men-plm-2026-western-jaguars-vs-daikyo-heroes/
  https://www.kenyahockeyunion.org/joomsport_match/premier-league-men-plm-2026-western-jaguars-vs-warriors/
  https://www.kenyahockeyunion.org/joomsport_match/premier-league-men-plm-2026-western-jaguars-vs-sikh-union-nairobi/


## Match Scraper

`scrape_match()` opens a single match page and extracts teams, final score, date/time, and the full event timeline (if one exists).

In [8]:
# Cell 8 - Pick a Test Match

# Grabs the first available match from whichever competition actually
# returned results, so the scraper test below works even if some
# competitions in SEASON_URLS came back empty.

first_match = None

for comp, links in all_match_links.items():
    if links:
        first_match = links[0]
        print(f"Using a {comp} match for testing:")
        print(first_match)
        break

if first_match is None:
    raise RuntimeError(
        "No match URLs were discovered for any competition — check "
        "SEASON_URLS and the discovery cell above before continuing."
    )


Using a PLW match for testing:
https://www.kenyahockeyunion.org/joomsport_match/premier-league-women-plw-2026-amira-sailors-hockey-club-vs-blazers-hockey-club/


In [9]:
# Cell 31 - Match Scraper Function (v2)

from bs4 import BeautifulSoup
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import TimeoutException

import re
import time


def scrape_match(match_url, competition="PLW", season=2026):

    driver.get(match_url)

    WebDriverWait(driver,20).until(
        EC.presence_of_element_located(
            (By.TAG_NAME,"body")
        )
    )

    # The match timeline table loads asynchronously, separately from the
    # rest of the page. A flat short pause here previously caused this
    # scraper to silently return an empty events list for any match where
    # the timeline hadn't finished rendering yet — indistinguishable from
    # a genuine scoreless match with no events, but wrong. Waiting
    # specifically for the timeline element (or confirming, via timeout,
    # that it genuinely never appears) fixes that.
    # Confirmed via direct verification against the live site: when a
    # match genuinely has no event timeline, no amount of waiting produces
    # one — it simply isn't in the page's data. A short wait here still
    # catches genuinely slow-loading timelines (rare), without paying a
    # full 10-second tax on every match that has none (the majority).
    try:
        WebDriverWait(driver, 3).until(
            EC.presence_of_element_located(
                (By.CLASS_NAME, "jsTblVerticalTimeLine")
            )
        )
    except TimeoutException:
        pass  # No timeline rendered within 3s — confirmed genuine no-events case, not a timing issue

    time.sleep(1)

    soup = BeautifulSoup(driver.page_source,"html.parser")

    # -----------------------
    # Teams
    # -----------------------

    home = soup.select_one(".jsMatchHomeTeam")
    away = soup.select_one(".jsMatchAwayTeam")

    if home is None or away is None:

        print("Could not find teams.")
        print(match_url)

        with open("debug.html","w",encoding="utf-8") as f:
            f.write(driver.page_source)

        raise Exception("Team names not found.")

    home_team = home.get_text(" ",strip=True)
    away_team = away.get_text(" ",strip=True)

    # -----------------------
    # Score
    # -----------------------

    score1 = soup.select_one(".BigMScore1")
    score2 = soup.select_one(".BigMScore2")

    home_score = int(score1.get_text(strip=True))
    away_score = int(score2.get_text(strip=True))

    # -----------------------
    # Date
    # -----------------------

    match_date = None
    match_time = None

    dt = soup.select_one(".matchdtime")

    if dt:

        parts = dt.get_text(" ",strip=True).split()

        if len(parts)>=2:
            match_date = parts[0]
            match_time = parts[1]

    # -----------------------
    # Match Number
    # -----------------------

    match_no = None

    content = soup.select_one(".jsMatchContentSection")

    if content:

        m = re.search(
            r"Match No\s*([0-9]+)",
            content.get_text("\n",strip=True)
        )

        if m:
            match_no = m.group(1)

    # -----------------------
    # Timeline
    # -----------------------

    events = []

    timeline = soup.select_one(".jsTblVerticalTimeLine")

    if timeline:

        for row in timeline.find_all("tr"):

            cols = row.find_all("td")

            if len(cols)<5:
                continue

            minute = cols[2].get_text(strip=True)

            if cols[1].find("img"):

                event = cols[1].find("img").get("title","")
                player = cols[0].get_text(" ",strip=True)
                team = home_team

            elif cols[3].find("img"):

                event = cols[3].find("img").get("title","")
                player = cols[4].get_text(" ",strip=True)
                team = away_team

            else:
                continue

            events.append({
                "minute": minute,
                "event": event,
                "player": player,
                "team": team
            })

    if (home_score + away_score) > 0 and len(events) == 0:
        print(f"  ⚠ {home_team} {home_score}-{away_score} {away_team}: score is nonzero "
              f"but no events were found even after waiting — worth a manual check of this match.")

    return {

        "competition": competition,
        "season": season,
        "match_no": match_no,
        "date": match_date,
        "time": match_time,
        "home_team": home_team,
        "away_team": away_team,
        "home_score": home_score,
        "away_score": away_score,
        "events": events,
        "url": match_url

    }

## Batch Scraping & Export

Run `scrape_match()` across every discovered match URL, assemble the results into `matches_df` / `events_df`, and save to `data/raw/`.

In [10]:
# Cell 32 - Test Complete Scraper

result = scrape_match(first_match)

print(result["home_team"])
print(result["away_team"])
print(result["home_score"])
print(result["away_score"])
print(len(result["events"]))

Amira Sailors Hockey Club
Blazers Hockey Club
0
3
3


In [11]:
# Cell 32B - Incremental Scraping Setup

import os
import pandas as pd

# Set to True to re-scrape every match from scratch (worth doing
# regularly — e.g. weekly — to catch KHU backfilling event data onto
# matches that previously had none. Incremental mode below won't detect
# that on its own, since it deliberately skips anything already saved.)
# Day-to-day runs should keep this False — re-scraping the entire season
# from zero every single time isn't sustainable as the match count grows.
FULL_RESCRAPE = False

existing_matches_df = pd.DataFrame()
existing_events_df = pd.DataFrame()
already_scraped_urls = set()

if not FULL_RESCRAPE and os.path.exists("../data/raw/matches_2026.csv"):

    existing_matches_df = pd.read_csv("../data/raw/matches_2026.csv")
    already_scraped_urls = set(existing_matches_df["URL"])

    if os.path.exists("../data/raw/events_2026.csv"):
        existing_events_df = pd.read_csv("../data/raw/events_2026.csv")

    print(f"Incremental mode: found {len(already_scraped_urls)} previously-scraped "
          f"match(es) — only new matches will be scraped this run.")

else:
    print("Full rescrape mode: every match will be scraped fresh, "
          "ignoring any previously saved data.")


Incremental mode: found 186 previously-scraped match(es) — only new matches will be scraped this run.


In [12]:
# Cell 33 - Scrape All Matches (Across All Competitions)

all_matches = []
failed_matches = []   # matches that failed even after retries, for a manual look afterward

# Incremental mode: only scrape matches not already in already_scraped_urls.
new_links = {
    comp: [url for url in links if url not in already_scraped_urls]
    for comp, links in all_match_links.items()
}

total = sum(len(v) for v in new_links.values())
already_known = sum(len(v) for v in all_match_links.values()) - total

if already_known > 0:
    print(f"{already_known} match(es) already scraped previously — skipping them this run.")
print(f"{total} new match(es) to scrape.\n")

done = 0

MAX_RETRIES = 3
RESTART_EVERY = 40   # proactively recycle the browser every N matches, since
                      # long Selenium sessions on Windows are prone to hanging
                      # or dying partway through a 100+ page scrape

matches_since_restart = 0

for comp, links in new_links.items():

    for url in links:

        done += 1
        print(f"[{comp}] {done}/{total}")

        # Proactive restart: avoids the kind of multi-hour session decay
        # that previously killed the browser partway through and caused
        # every remaining match to fail instantly.
        if matches_since_restart >= RESTART_EVERY:
            print("  ↻ Proactively restarting the browser to avoid session decay...")
            try:
                driver.quit()
            except Exception:
                pass
            driver = create_driver()
            matches_since_restart = 0

        success = False

        for attempt in range(1, MAX_RETRIES + 1):

            try:

                data = scrape_match(url, competition=comp, season=2026)

                all_matches.append(data)

                print(
                    f"✓ {data['home_team']} {data['home_score']} - {data['away_score']} {data['away_team']}"
                )

                success = True
                matches_since_restart += 1
                break

            except Exception as e:

                err = str(e)
                session_dead = (
                    "invalid session id" in err
                    or "session deleted" in err
                    or "chrome not reachable" in err
                    or "Read timed out" in err
                    or "Connection refused" in err
                )

                if session_dead:
                    print(f"  ⚠ Attempt {attempt}/{MAX_RETRIES}: browser session appears dead ({err.splitlines()[0]}). Restarting...")
                    try:
                        driver.quit()
                    except Exception:
                        pass
                    driver = create_driver()
                    matches_since_restart = 0
                    time.sleep(2)
                else:
                    print(f"  ⚠ Attempt {attempt}/{MAX_RETRIES} failed: {err.splitlines()[0]}")
                    time.sleep(2)

        if not success:
            print(f"✗ Giving up on this match after {MAX_RETRIES} attempts: {url}")
            failed_matches.append({"competition": comp, "url": url})

print("\nFinished this run")
print("New matches scraped:", len(all_matches))
print("Matches that failed after retries:", len(failed_matches))

if failed_matches:
    print("\nFailed matches (re-run Cell 33B below to retry just these):")
    for f in failed_matches:
        print(f"  [{f['competition']}] {f['url']}")


186 match(es) already scraped previously — skipping them this run.
0 new match(es) to scrape.


Finished this run
New matches scraped: 0
Matches that failed after retries: 0


In [13]:
# Cell 33B - Retry Failed Matches (run only if failed_matches is non-empty)

# If Cell 33 above finished with some matches in failed_matches (e.g. a
# genuinely broken page, not just a dead browser session), this retries
# just those — no need to re-scrape all 130+ matches to pick up a
# handful of stragglers.

if not failed_matches:
    print("No failed matches to retry.")
else:
    still_failed = []

    for f in list(failed_matches):

        comp, url = f["competition"], f["url"]
        print(f"Retrying [{comp}] {url}")

        try:
            data = scrape_match(url, competition=comp, season=2026)
            all_matches.append(data)
            print(f"✓ {data['home_team']} {data['home_score']} - {data['away_score']} {data['away_team']}")

        except Exception as e:
            print(f"  ✗ Still failing: {str(e).splitlines()[0]}")
            try:
                driver.quit()
            except Exception:
                pass
            driver = create_driver()
            still_failed.append(f)

    failed_matches = still_failed

    print()
    print("Matches scraped so far:", len(all_matches))
    print("Still failing:", len(failed_matches))

    if failed_matches:
        print("\nThese matches still need attention (check the URL manually in a browser):")
        for f in failed_matches:
            print(f"  [{f['competition']}] {f['url']}")


No failed matches to retry.


In [14]:
# Cell 34 - Match DataFrame

new_matches_df = pd.DataFrame([

    {
        "Competition": m["competition"],
        "Season": m["season"],
        "MatchNo": m["match_no"],
        "Date": m["date"],
        "Time": m["time"],
        "HomeTeam": m["home_team"],
        "AwayTeam": m["away_team"],
        "HomeGoals": m["home_score"],
        "AwayGoals": m["away_score"],
        "URL": m["url"]
    }

    for m in all_matches

])

# Merge with previously-scraped matches (empty in full-rescrape mode).
# drop_duplicates on URL is a safety net — shouldn't ever trigger given
# the skip logic above, but guarantees no match is ever double-counted.
matches_df = pd.concat([existing_matches_df, new_matches_df], ignore_index=True)
matches_df = matches_df.drop_duplicates(subset="URL", keep="last").reset_index(drop=True)

print(f"Matches this run: {len(new_matches_df)} new. "
      f"Total after merging with previously-scraped data: {len(matches_df)}.")

matches_df.head()


Matches this run: 0 new. Total after merging with previously-scraped data: 186.


,Competition,Season,MatchNo,Date,Time,HomeTeam,AwayTeam,HomeGoals,AwayGoals,URL
0,PLW,2026,14.0,24-05-2026,14:00,Amira Sailors Hockey Club,Blazers Hockey Club,0,3,https://www.kenyahockeyunion.org/joomsport_mat...
1,PLW,2026,NaN,01-06-2026,15:00,Amira Sailors Hockey Club,Sliders Hockey Club,0,0,https://www.kenyahockeyunion.org/joomsport_mat...
2,PLW,2026,1.0,23-05-2026,12:00,Blazers Hockey Club,Kenyatta University Ladies,2,0,https://www.kenyahockeyunion.org/joomsport_mat...
3,PLW,2026,NaN,04-07-2026,15:00,Blazers Hockey Club,Kisumu Queens,1,0,https://www.kenyahockeyunion.org/joomsport_mat...
4,PLW,2026,40.0,06-06-2026,15:00,Blazers Hockey Club,Sliders Hockey Club,3,0,https://www.kenyahockeyunion.org/joomsport_mat...


In [15]:
# Cell 35 - Events DataFrame

new_events = []

for match in all_matches:

    for e in match["events"]:

        new_events.append({

            "Competition": match["competition"],
            "Season": match["season"],
            "MatchNo": match["match_no"],
            "Date": match["date"],
            "HomeTeam": match["home_team"],
            "AwayTeam": match["away_team"],
            "Minute": e["minute"],
            "Event": e["event"],
            "Player": e["player"],
            "Team": e["team"]

        })

new_events_df = pd.DataFrame(new_events)

# Events only exist for newly-scraped matches here, so a straight concat
# is safe — no duplication risk, since already-scraped matches were
# skipped entirely and never re-added to all_matches.
events_df = pd.concat([existing_events_df, new_events_df], ignore_index=True)

print(f"Events this run: {len(new_events_df)} new. "
      f"Total after merging with previously-scraped data: {len(events_df)}.")

events_df.head()


Events this run: 0 new. Total after merging with previously-scraped data: 209.


,Competition,Season,MatchNo,Date,HomeTeam,AwayTeam,Minute,Event,Player,Team
0,PLW,2026,14.0,24-05-2026,Amira Sailors Hockey Club,Blazers Hockey Club,16',Goal,Audrey Omaido,Blazers Hockey Club
1,PLW,2026,14.0,24-05-2026,Amira Sailors Hockey Club,Blazers Hockey Club,42',Goal,Joan Ajao,Blazers Hockey Club
2,PLW,2026,14.0,24-05-2026,Amira Sailors Hockey Club,Blazers Hockey Club,44',Goal,Amanda Ijai,Blazers Hockey Club
3,PLW,2026,1.0,23-05-2026,Blazers Hockey Club,Kenyatta University Ladies,27',Green Card,Joan Ajao,Blazers Hockey Club
4,PLW,2026,1.0,23-05-2026,Blazers Hockey Club,Kenyatta University Ladies,36',Goal,Joan Ajao,Blazers Hockey Club


In [16]:
# Cell 36 - Check Data

print("Matches:", len(matches_df))
print("Events :", len(events_df))

# Sanity check against the FULL merged dataset (not just this run's new
# matches) — a competition having zero *new* matches this run is normal
# (nothing new was played there since last scrape), not a warning-worthy
# condition. This only warns if a competition has genuinely never been
# scraped at all, across every run to date.
scraped_by_comp = matches_df["Competition"].value_counts()
print()
print("Total matches per competition (all-time):")
print(scraped_by_comp)

missing = set(SEASON_URLS) - set(scraped_by_comp.index)
if missing:
    print(f"\n⚠ No matches have ever been scraped for: {sorted(missing)} — check SEASON_URLS for these.")

matches_df.head()


Matches: 186
Events : 209

Total matches per competition (all-time):
Competition
PLM       52
SLM       33
NLM-EZ    31
PLW       30
SLW       20
NLM-WZ    11
NLM-SZ     6
NLM-CZ     3
Name: count, dtype: int64


,Competition,Season,MatchNo,Date,Time,HomeTeam,AwayTeam,HomeGoals,AwayGoals,URL
0,PLW,2026,14.0,24-05-2026,14:00,Amira Sailors Hockey Club,Blazers Hockey Club,0,3,https://www.kenyahockeyunion.org/joomsport_mat...
1,PLW,2026,NaN,01-06-2026,15:00,Amira Sailors Hockey Club,Sliders Hockey Club,0,0,https://www.kenyahockeyunion.org/joomsport_mat...
2,PLW,2026,1.0,23-05-2026,12:00,Blazers Hockey Club,Kenyatta University Ladies,2,0,https://www.kenyahockeyunion.org/joomsport_mat...
3,PLW,2026,NaN,04-07-2026,15:00,Blazers Hockey Club,Kisumu Queens,1,0,https://www.kenyahockeyunion.org/joomsport_mat...
4,PLW,2026,40.0,06-06-2026,15:00,Blazers Hockey Club,Sliders Hockey Club,3,0,https://www.kenyahockeyunion.org/joomsport_mat...


In [17]:
# Cell 37 - Save Matches

matches_df.to_csv(
    "../data/raw/matches_2026.csv",
    index=False
)

print("Matches saved.")

Matches saved.


In [18]:
# Cell 38 - Save Events

events_df.to_csv(
    "../data/raw/events_2026.csv",
    index=False
)

print("Events saved.")

Events saved.


## Team Aggregate Player Stats (New — richer than match-timeline data)

Confirmed by direct inspection of two real team pages (Butali Warriors,
Sikh Union Nairobi) before writing any code: each team\'s own page on
kenyahockeyunion.org has a **"Players Stats" table separate from match
timelines** — an all-time aggregate of goals/cards per player for that
club, built from KHU\'s own internal records rather than reconstructed
from individual match event timelines.

**Why this matters:** the domestic Data Integrity Check earlier in this
project confirmed only ~9.5% of matches have event-level (player-name)
detail in their timelines. This team-page table appears to be a
separate, more consistently maintained system — real examples showed
players with 10-12 goals, far above anything the timeline-based
top-scorers table has ever produced. If this holds at scale, it\'s a
genuinely better source for domestic top scorers.

**Confirmed distinguishing rule, verified on two teams:** each team page
actually has **two** stats tables — one scoped to whichever season is
currently selected (columns: Goal, Yellow Card, Green Card, Shirt
Number), and one all-time aggregate across every season the club has
data for (columns: Goal, Yellow Card, Green Card, **Red Card**, Shirt
Number — note the extra Red Card column). This module deliberately
scrapes the **all-time aggregate** table specifically, identified by
that extra column, not by position on the page.

**Honest limitation, stated directly:** this is an all-time career
total per club, not scoped to the current season the way the rest of
this project\'s domestic data is. It also inherits a real data-quality
issue confirmed during reconnaissance — the same player sometimes
appears twice under slightly different name spellings or separate
per-season profile records (e.g. two "Edgar Juma" entries, "Harvir
Singh Ghataurae" vs "Harvir Ghathaure"), not automatically merged by
KHU\'s own system. This is reported as found, not silently deduplicated.


In [19]:
# Cell 39 - Discover All Team Pages

# Reuses the driver already open above. For each competition, opens its
# season page and collects every unique team page link found on it —
# the same reliable "read real links off a real page" approach used for
# match discovery, rather than trying to guess team URL slugs from team
# names (confirmed unreliable: "Butali Warriors" -> "butali-warriors",
# but the page title just shows "Warriors" — not a predictable pattern).

team_pages = {}   # team display name -> team page URL

for comp_key, season_url in SEASON_URLS.items():

    print(f"Discovering teams in {comp_key}...")

    driver.get(season_url)
    time.sleep(5)

    soup = BeautifulSoup(driver.page_source, "html.parser")

    for a in soup.find_all("a", href=True):
        if "/joomsport_team/" not in a["href"]:
            continue
        name = a.get_text(strip=True)
        if not name:
            continue
        # strip any ?sid=... query parameter — we want the team\'s base
        # page, not one filtered to a specific season
        base_url = a["href"].split("?")[0]
        if not base_url.startswith("http"):
            base_url = f"https://www.kenyahockeyunion.org{base_url}"
        team_pages[name] = base_url

print(f"\nDiscovered {len(team_pages)} unique team pages across all competitions.")


Discovering teams in PLW...


Discovering teams in PLM...


Discovering teams in SLM...


Discovering teams in SLW...


Discovering teams in NLM-EZ...


Discovering teams in NLM-CZ...


Discovering teams in NLM-WZ...


Discovering teams in NLM-SZ...



Discovered 75 unique team pages across all competitions.


In [20]:
# Cell 40 - Team Player Stats Scraper Function

def scrape_team_player_stats(team_url, team_name):
    """
    Fetches a team\'s page and extracts its Players Stats table.

    Confirmed by direct diagnostic against a real page (Blazers Hockey
    Club): the goal/card stat columns use IMAGE ICONS as headers, not
    text \u2014 pandas reads these as "Unnamed: 1", "Unnamed: 2", etc.,
    never as "Goal" or "Red Card". A real page also contains several
    OTHER tables (other competitions\' standings) that must be excluded.
    Matching on "Name" + "Shirt Number" \u2014 both confirmed present as
    real text headers \u2014 reliably identifies the player table without
    depending on icon-column text that will never exist.

    Goals are read from the first stat column after "Name" by position,
    not by column name, since that name is never real text on this site.
    """
    driver.get(team_url)

    try:
        WebDriverWait(driver, 15).until(
            EC.presence_of_element_located((By.TAG_NAME, "body"))
        )
    except TimeoutException:
        print(f"  \u26a0 Page did not load in time: {team_url}")
        return []

    try:
        WebDriverWait(driver, 10).until(
            EC.presence_of_element_located((By.TAG_NAME, "table"))
        )
    except TimeoutException:
        pass

    time.sleep(0.5)

    html = driver.page_source

    try:
        tables = pd.read_html(io.StringIO(html), flavor="lxml")
    except (ValueError, ImportError):
        return []

    candidate_tables = []
    for t in tables:
        cols = [str(c).strip().lower() for c in t.columns]
        has_name = "name" in cols
        has_shirt = any("shirt" in c for c in cols)
        if has_name and has_shirt:
            candidate_tables.append(t)

    if not candidate_tables:
        return []

    # If more than one matches (some team pages show both a season view
    # and an all-time view), prefer whichever has the higher total in
    # its first stat column \u2014 the more complete/cumulative one.
    def first_stat_col_sum(t):
        cols = list(t.columns)
        name_idx = [i for i, c in enumerate(cols) if str(c).strip().lower() == "name"][0]
        stat_idx = name_idx + 1
        if stat_idx < len(cols):
            return pd.to_numeric(t.iloc[:, stat_idx], errors="coerce").fillna(0).sum()
        return 0

    target_table = max(candidate_tables, key=first_stat_col_sum)

    cols = list(target_table.columns)
    name_idx = [i for i, c in enumerate(cols) if str(c).strip().lower() == "name"][0]
    goals_idx = name_idx + 1

    if goals_idx >= len(cols):
        return []

    records = []
    for _, row in target_table.iterrows():
        name_text = str(row.iloc[name_idx]).strip()
        if not name_text or name_text.lower() == "nan":
            continue
        # Strip trailing duplicate name text picked up from the image
        # alt-text + link text pairing seen in real pages
        # (e.g. "Festus OnyangoFestus Onyango")
        half = len(name_text) // 2
        if len(name_text) % 2 == 0 and name_text[:half] == name_text[half:]:
            name_text = name_text[:half]

        try:
            goals = int(row.iloc[goals_idx]) if pd.notna(row.iloc[goals_idx]) else 0
        except (ValueError, TypeError):
            goals = 0

        records.append({
            "Team": team_name,
            "Player": name_text,
            "Goals": goals,
            "URL": team_url,
        })

    return records


print("scrape_team_player_stats() ready.")


scrape_team_player_stats() ready.


In [21]:
# DIAGNOSTIC - Team Page Table Structure
# Run this on ONE real team page (e.g. Butali Warriors, confirmed to
# have real data) if the coverage summary above still shows 0 teams
# found. Paste the full output back for a precise fix, the same way
# earlier scraping issues in this project were diagnosed and resolved.

test_team_url = list(team_pages.values())[0]
test_team_name = list(team_pages.keys())[0]
print("Testing:", test_team_name, "-", test_team_url)

driver.get(test_team_url)
try:
    WebDriverWait(driver, 15).until(EC.presence_of_element_located((By.TAG_NAME, "body")))
except TimeoutException:
    pass

try:
    WebDriverWait(driver, 10).until(EC.presence_of_element_located((By.TAG_NAME, "table")))
    print("A <table> element appeared on the page.")
except TimeoutException:
    print("\u26a0 No <table> element ever appeared within 10 seconds.")

time.sleep(0.5)
html = driver.page_source

try:
    tables = pd.read_html(io.StringIO(html), flavor="lxml")
    print(f"\npd.read_html found {len(tables)} table(s)\n")
    for idx, t in enumerate(tables):
        print(f"=== TABLE {idx} ===")
        print("Columns:", list(t.columns))
        print(t.head(3))
        print()
except (ValueError, ImportError) as e:
    print("pd.read_html found no tables at all:", e)


Testing: Blazers Hockey Club - https://www.kenyahockeyunion.org/joomsport_team/blazers-hockey-club/


A <table> element appeared on the page.



pd.read_html found 9 table(s)

=== TABLE 0 ===
Columns: ['Name', 'Unnamed: 1', 'Unnamed: 2', 'Unnamed: 3', 'Shirt Number']
              Name  Unnamed: 1  Unnamed: 2  Unnamed: 3  Shirt Number
0    Alice Wanjiru           0           0           0           NaN
1      Amanda Ijai           1           0           0           NaN
2  Angeline Oketch           0           0           0           NaN

=== TABLE 1 ===
Columns: ['#', 'Teams', 'Pl', 'Pts']
   #               Teams  Pl  Pts
0  1     Western Jaguars  11   23
1  2            Warriors  10   19
2  3  Sikh Union Nairobi  10   18

=== TABLE 2 ===
Columns: ['#', 'Teams', 'Pl', 'Pts']
   #                Teams  Pl  Pts
0  1  Blazers Hockey Club   7   17
1  2      USIU – A Ladies   8   13
2  3   Lakers Hockey Club   6   11

=== TABLE 3 ===
Columns: ['#', 'Teams', 'Pl', 'Pts']
   #                  Teams  Pl  Pts
0  1         KCA University   8   18
1  2  Strathmore University   6   14
2  3               Mvita XI   5   11

=== TABLE 4 =

In [22]:
# Cell 41 - Scrape Player Stats for All Discovered Teams

# Same resilience pattern already proven for domestic match scraping
# (Cell 33): long-running Selenium sessions can degrade after enough
# requests in a row, surfacing as network-level errors like
# "net::ERR_NAME_NOT_RESOLVED" that have nothing to do with the target
# page itself — confirmed by a real run that failed this way on the
# 20th team in a row. The fix is the same: retry with driver recreation
# on failure, and proactively restart the browser periodically so it
# never gets the chance to degrade that far in the first place.

all_team_player_stats = []
teams_with_no_table = []
failed_teams = []

MAX_RETRIES = 3
RESTART_EVERY = 20
teams_since_restart = 0

start_time = time.time()

for i, (team_name, team_url) in enumerate(team_pages.items(), start=1):
    elapsed = time.time() - start_time
    avg_per_team = elapsed / (i - 1) if i > 1 else None
    remaining_str = ""
    if avg_per_team is not None:
        remaining = avg_per_team * (len(team_pages) - i + 1)
        remaining_str = f" \u2014 est. {remaining/60:.1f} min remaining"

    print(f"[{i}/{len(team_pages)}] {team_name}{remaining_str}")

    if teams_since_restart >= RESTART_EVERY:
        print("  \u21bb Proactively restarting the browser to avoid session decay...")
        try:
            driver.quit()
        except Exception:
            pass
        driver = create_driver()
        teams_since_restart = 0

    success = False
    records = []

    for attempt in range(1, MAX_RETRIES + 1):
        try:
            records = scrape_team_player_stats(team_url, team_name)
            success = True
            teams_since_restart += 1
            break
        except Exception as e:
            err = str(e)
            session_dead = (
                "ERR_NAME_NOT_RESOLVED" in err
                or "invalid session id" in err
                or "session deleted" in err
                or "chrome not reachable" in err
                or "Read timed out" in err
                or "Connection refused" in err
            )
            if session_dead:
                print(f"  \u26a0 Attempt {attempt}/{MAX_RETRIES}: browser/network issue ({err.splitlines()[0][:80]}). Restarting driver...")
                try:
                    driver.quit()
                except Exception:
                    pass
                driver = create_driver()
                teams_since_restart = 0
                time.sleep(2)
            else:
                print(f"  \u26a0 Attempt {attempt}/{MAX_RETRIES} failed: {err.splitlines()[0][:80]}")
                time.sleep(2)

    if not success:
        print(f"  \u2717 Giving up on {team_name} after {MAX_RETRIES} attempts.")
        failed_teams.append((team_name, team_url))
    elif records:
        all_team_player_stats.extend(records)
        top = max(records, key=lambda r: r["Goals"])
        top_name = top["Player"]
        top_goals = top["Goals"]
        print(f"  \u2713 {len(records)} player(s) \u2014 top scorer: {top_name} ({top_goals} goals)")
    else:
        teams_with_no_table.append(team_name)
        print(f"  \u26a0 No all-time stats table found for this team")

total_elapsed = time.time() - start_time
print()
print("=" * 70)
print("TEAM PLAYER STATS COVERAGE")
print("=" * 70)
print(f"Total time              : {total_elapsed/60:.1f} minutes for {len(team_pages)} teams")
print(f"Teams attempted        : {len(team_pages)}")
print(f"Teams with data found  : {len(team_pages) - len(teams_with_no_table) - len(failed_teams)}")
print(f"Teams with no table    : {len(teams_with_no_table)}")
print(f"Teams that failed      : {len(failed_teams)}")
if teams_with_no_table:
    print("\nTeams with no table found (may need manual URL check):")
    for t in teams_with_no_table:
        print(f"  - {t}")
if failed_teams:
    print("\nTeams that failed even after retries (re-run Cell 41B below to retry just these):")
    for t, u in failed_teams:
        print(f"  - {t}: {u}")


[1/75] Blazers Hockey Club


  ✓ 26 player(s) — top scorer: Carol Guchu (6 goals)
[2/75] USIU – A Ladies — est. 1.5 min remaining


  ✓ 30 player(s) — top scorer: Lynn Mwangi (5 goals)
[3/75] Lakers Hockey Club — est. 1.4 min remaining


  ✓ 25 player(s) — top scorer: Alice Owiti (3 goals)
[4/75] Strathmore University Ladies — est. 1.4 min remaining


  ✓ 26 player(s) — top scorer: Grace Bwire (4 goals)
[5/75] Amira Sailors Hockey Club — est. 1.3 min remaining


...

  ⚠ No all-time stats table found for this team
[72/75] Kabianga University — est. 0.1 min remaining


  ⚠ No all-time stats table found for this team
[73/75] Kisii University — est. 0.1 min remaining


  ✓ 1 player(s) — top scorer: Tonny Rogers (0 goals)
[74/75] Rongo University — est. 0.0 min remaining


  ⚠ No all-time stats table found for this team
[75/75] Oyugis Hockey Club — est. 0.0 min remaining


  ⚠ No all-time stats table found for this team

TEAM PLAYER STATS COVERAGE
Total time              : 1.4 minutes for 75 teams
Teams attempted        : 75
Teams with data found  : 51
Teams with no table    : 24
Teams that failed      : 0

Teams with no table found (may need manual URL check):
  - Ulinzi Patriots
  - GHF Rift Pirates Men
  - Thika Rovers
  - Dukes of Gloucester
  - Fire Flickers
  - Kisii Falcons
  - Bay Club
  - Gorillas Hockey CLub – Migori
  - Impala
  - GHF Blue Pirates
  - UOE Men
  - TUK Men
  - Snippers Hockey Club (Makueni)
  - Chuka University
  - Egerton Uni Sharks
  - Maasai Mara Hockey Team
  - Mangu Legends
  - Nyari Hockey Club
  - Nandi Hawks
  - GHF Blue Pirates II
  - Western Jaguars Dev
  - Kabianga University
  - Rongo University
  - Oyugis Hockey Club


In [23]:
# Cell 41B - Retry Failed Teams (run only if failed_teams is non-empty)

if len(failed_teams) > 0:
    print(f"Retrying {len(failed_teams)} team(s) that failed after the main run...")
    still_failed = []

    for team_name, team_url in failed_teams:
        print(f"Retrying: {team_name}")
        try:
            records = scrape_team_player_stats(team_url, team_name)
            if records:
                all_team_player_stats.extend(records)
                print(f"  \u2713 Recovered \u2014 {len(records)} player(s) found")
            else:
                teams_with_no_table.append(team_name)
                print("  \u26a0 No table found on retry")
        except Exception as e:
            print(f"  \u2717 Still failing: {str(e).splitlines()[0][:80]}")
            still_failed.append((team_name, team_url))

    failed_teams = still_failed
    print(f"\n{len(failed_teams)} team(s) still failing after retry.")
else:
    print("No failed teams to retry.")


No failed teams to retry.


In [24]:
# Cell 42 - Save Team Player Stats

# Explicit columns, even when all_team_player_stats is empty — confirmed
# by a real run that this matters: pd.DataFrame([]) produces zero
# COLUMNS, not just zero rows, and saving that to CSV creates a file
# with no header at all, which pandas cannot read back
# ("EmptyDataError: No columns to parse from file") even though the
# file technically exists. This keeps the saved file always valid,
# regardless of whether any teams were successfully scraped this run.

team_player_stats_df = pd.DataFrame(
    all_team_player_stats,
    columns=["Team", "Player", "Goals", "URL"]
)

team_player_stats_df.to_csv("../data/raw/team_player_stats.csv", index=False)

if len(team_player_stats_df) == 0:
    print("\u26a0 Saved team_player_stats.csv, but it has zero records \u2014 "
          "every team either failed or had no all-time stats table this run. "
          "Check the coverage summary above for details.")
else:
    print(f"Saved team_player_stats.csv \u2014 {len(team_player_stats_df)} player-team records "
          f"across {team_player_stats_df['Team'].nunique()} teams.")


Saved team_player_stats.csv — 867 player-team records across 51 teams.
